####  Notebook Documentation

#### 📝 Notebook Details

- **Notebook Name:** `01_NB_Oracle_OneTimeSilverLoad`  
- **Created By:** Fabric Accelerator  
- **Created Date:** 15 May 2025  
- **Last Modified By:** Fabric Accelerator  
- **Last Modified Date:** 24 May 2025  

---

#### 📌 Purpose

This notebook is designed to perform a **one-time data load** into the **Oracle Silver Layer** as part of the data processing pipeline.

---

#### 📎 Notes

- Intended for initial data migration or backfill scenarios  
- Not designed for recurring or scheduled execution  

In [1]:
############################################# IMPORTING REQUIRED PACKAGES ##############################################
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
col, lit, concat, when, trim, collect_list, row_number, to_date,date_format,
col, concat, lit, to_timestamp, date_format, to_date, coalesce,concat_ws, md5,
udf , concat, to_timestamp,count, when )
from pyspark.sql.window import Window
from datetime import datetime, timezone, timedelta
from delta.tables import DeltaTable
import pandas as pd
from com.microsoft.spark.fabric import Constants
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants
from pyspark.sql.types import (
    DoubleType, IntegerType, BooleanType, StringType,StructField,ArrayType,
    BinaryType, TimestampType, DateType, LongType, StructType
)
import concurrent.futures
import pytz
import traceback
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when
from pyspark.sql.functions import col, sum as _sum
import json
import requests

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 3, Finished, Available, Finished, False)

In [2]:
################################################### NOTEBOOK PARAMETERS ####################################################
ETLBatchId = -1
ConfigSchemaName = 'Config_Oracle'

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 4, Finished, Available, Finished, False)

In [3]:
################################################### BUILDING CUSTOM SPARK SESSION ####################################################
spark = SparkSession.builder \
    .appName("OracleDataLoad") \
    .config("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .config("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED") \
    .config("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED") \
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED") \
    .config("spark.gluten.enabled", "false") \
    .config("spark.sql.legacy.parquet.datetimeRebaseModeInWrite", "CORRECTED") \
    .config("spark.sql.legacy.parquet.datetimeRebaseModeInRead", "CORRECTED") \
    .config("spark.sql.caseSensitive", "true") \
    .config("spark.sql.extensions", "delta.sql.DeltaSparkSessionExtensions") \
    .config("spark.sql.catalog.spark_catalog", "delta.catalog.DeltaCatalog") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")\
    .getOrCreate()

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 5, Finished, Available, Finished, False)

In [4]:
################################################### GETTING SPARK CONFIG DETAILS ####################################################
config_value = spark.conf.get("spark.executor.memoryOverhead", "Not Set")
config = spark.conf.get("spark.executor.memory", "Not Set")

print(f"spark.executor.memoryOverhead: {config_value}")
print(f"spark.executor.memory: {config}")

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 6, Finished, Available, Finished, False)

spark.executor.memoryOverhead: 384m
spark.executor.memory: 56g


In [5]:
################################################### USER DEFINED VARIABLES ####################################################
current_utc_time = datetime.now(timezone.utc)
user_name = mssparkutils.env.getUserName()

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 7, Finished, Available, Finished, False)

In [6]:
################################################### LOADING METADATA ####################################################
try:
    ConfigQuery = f"SELECT * FROM {ConfigSchemaName}.OneTimeConfigETL"
    InformationSchema=f"SELECT * FROM {ConfigSchemaName}.SourceInformationSchema"
    ConfigETL_df= spark.read.option(Constants.DatabaseName, "WH_MetaData").synapsesql(ConfigQuery) 
    InformationSchema_df= spark.read.option(Constants.DatabaseName, "WH_MetaData").synapsesql(InformationSchema) 
    source_table_names = [row.SourceTableName for row in ConfigETL_df.select("SourceTableName").collect()]
    InformationSchema_df = InformationSchema_df.filter(col("TABLE_NAME").isin(source_table_names))
except Exception as e:
    print(e)

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 8, Finished, Available, Finished, False)

In [7]:
ConfigETL_df = ConfigETL_df.filter(col("Id")==8)

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 9, Finished, Available, Finished, False)

In [8]:
display(ConfigETL_df)

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e02a5b17-addd-4b6b-ab7c-f51dcd632b93)

In [9]:
############################################################ DEFINING ABFSS PATH ##########################################################################
token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")
headers = {"Authorization": f"Bearer {token}"}
workspace_id = mssparkutils.runtime.context.get("currentWorkspaceId")
if not workspace_id:
    raise Exception("No workspace context found. Attach notebook to a Lakehouse.")
print(f"Workspace ID: {workspace_id}")
url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items"
items = requests.get(url, headers=headers).json().get("value", [])
lakehouses = [i for i in items if i["type"].lower() == "lakehouse"]
warehouses = [i for i in items if i["type"].lower() == "warehouse"]
bronze_lh = next((lh for lh in lakehouses if lh["displayName"].upper().startswith("LH_BRONZE")), None)
silver_lh = next((lh for lh in lakehouses if lh["displayName"].upper().startswith("LH_SILVER")), None)
if not bronze_lh:
    raise Exception("LH_BRONZE lakehouse not found in workspace")
if not silver_lh:
    raise Exception(" LH_SILVER lakehouse not found in workspace")
print(f" Bronze Lakehouse: {bronze_lh['displayName']}")
print(f" Silver Lakehouse: {silver_lh['displayName']}")
warehouse = next((wh for wh in warehouses if wh["displayName"].upper().startswith("WH_METADATA")), None)
if not warehouse:
    raise Exception(" WH_MetaData warehouse not found in workspace")
print(f" Warehouse: {warehouse['displayName']}")
workspace_name = mssparkutils.runtime.context.get("currentWorkspaceName")
bronze_base_path = f"abfss://{workspace_name}@onelake.dfs.fabric.microsoft.com/{bronze_lh['displayName']}.Lakehouse"
silver_base_path = f"abfss://{workspace_name}@onelake.dfs.fabric.microsoft.com/{silver_lh['displayName']}.Lakehouse/Tables"
warehouse_name = warehouse["displayName"]
WAREHOUSE_BATCH = f"{warehouse_name}.Log.ETLBatchHeader"
WAREHOUSE_SILVER = f"{warehouse_name}.Log.ETLBatchSilverDetails"
WAREHOUSE_CONFIG = f"{warehouse_name}.{ConfigSchemaName}.OneTimeConfigETL"
################################################## OUTPUTS ###################################################################################################
print("\n Final Configurations:")
print(f"Bronze Path: {bronze_base_path}")
print(f"Silver Path: {silver_base_path}")
print(f"Batch Table: {WAREHOUSE_BATCH}")
print(f"Silver Log Table: {WAREHOUSE_SILVER}")
print(f"Config Table: {WAREHOUSE_CONFIG}")

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 11, Finished, Available, Finished, False)

Workspace ID: e6a9c954-f246-41d9-a086-ef657a5c18cd
 Bronze Lakehouse: LH_BRONZE
 Silver Lakehouse: LH_SILVER
 Warehouse: WH_MetaData

 Final Configurations:
Bronze Path: abfss://OracleToFabricRTI@onelake.dfs.fabric.microsoft.com/LH_BRONZE.Lakehouse
Silver Path: abfss://OracleToFabricRTI@onelake.dfs.fabric.microsoft.com/LH_SILVER.Lakehouse/Tables
Batch Table: WH_MetaData.Log.ETLBatchHeader
Silver Log Table: WH_MetaData.Log.ETLBatchSilverDetails
Config Table: WH_MetaData.Config_Oracle.OneTimeConfigETL


In [10]:
abfss://OracleToFabricRTI@onelake.dfs.fabric.microsoft.com/LH_BRONZE.Lakehouse/Files/HOSPITAL/ALL_DATATYPES_TEST/data_6d5abf90-59b9-4346-8b07-d4e26fe78053_aa8737b2-1c2b-442e-924d-aae340d419ed.parquet

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 12, Finished, Available, Finished, False)

SyntaxError: invalid decimal literal (3139046719.py, line 1)

In [11]:
################################################### USER DEFINED FUNCTION: 1.EXTRACTING BRONZE DATA ####################################################
def Read_bronze_data(BronzeSchemaName, BronzeTableName,read_type):
    is_okay = 1
    error_message = None
    msg = ''
    # read_type = 'Table'
    base_path = bronze_base_path
    if read_type == 'Table':
        try:
            bronze_abfss_path = f"{base_path}/Tables/{BronzeSchemaName}/{BronzeTableName}"
            # bronze_df = spark.read.format('delta').load(bronze_abfss_path)
            bronze_df = spark.read.format('delta').load(bronze_abfss_path)    
            bronze_df = bronze_df.drop_duplicates()
            msg = f"Data has been successfully extracted for Schema Name {BronzeSchemaName}, Table Name: {BronzeTableName}"
        except Exception as e:
            msg = f"An error occurred while extracting data for Schema Name: {BronzeSchemaName}, Table Name: {BronzeTableName}"
            error_message = str(e)
            print(error_message)
            bronze_df = None
            is_okay  = 0
            return is_okay, bronze_df, msg, error_message
    else:
        if read_type == 'File':
            print(read_type)
            try:
                bronze_abfss_path = f"{base_path}/Files/{BronzeSchemaName}/{BronzeTableName}/*.parquet"
                print(bronze_abfss_path)
                bronze_df = spark.read \
                .option("spark.sql.parquet.inferTimestampNTZ.enabled","false") \
                .parquet(bronze_abfss_path)
                # bronze_df = spark.read.format('parquet').load(bronze_abfss_path)
                print("Data REad")    
                bronze_df = bronze_df.drop_duplicates()
                msg = f"Data has been successfully extracted for Schema Name {BronzeSchemaName}, Table Name: {BronzeTableName}"
            except Exception as e:
                msg = f"An error occurred while extracting data for Schema Name: {BronzeSchemaName}, Table Name: {BronzeTableName}"
                error_message = str(e)
                bronze_df = None
                print(error_message)
                is_okay  = 0
                return is_okay, bronze_df, msg, error_message
    return is_okay, bronze_df, msg, error_message

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 13, Finished, Available, Finished, False)

In [12]:
################################################### USER DEFINED FUNCTION: 2.DATA TYPE CASTING ####################################################
def data_type_casting(table_name, bronze_df, config_mapping_schema_df):
    is_okay = 1
    error_message = None
    msg = 'Data Type Casting was Successful'  

    data_type_mapping = {
        'VARCHAR2': StringType(),
        'TIMESTAMP(6) WITH TIME ZONE': TimestampType(),
        'DATE': TimestampType(),
        'NVARCHAR2': StringType(),
        'NUMBER': DoubleType(),
        'CHAR': StringType(),
        'RAW':StringType(),
        'CLOB':StringType(),
        'XMLTYPE':StringType()
    }

    try:
        distinct_tables = [row['TABLE_NAME'] for row in config_mapping_schema_df.select("TABLE_NAME").distinct().toLocalIterator()]
        print(f"Data read completed for {table_name}")

        if table_name in distinct_tables:
            table_config = config_mapping_schema_df.filter(col("TABLE_NAME") == table_name)

            table_config_sorted = table_config.orderBy("COLUMN_ID")

            for row in table_config_sorted.collect():
                column_name = row['COLUMN_NAME']
                data_type = row['DATA_TYPE']
                
                if column_name in bronze_df.columns:
                    spark_data_type = data_type_mapping.get(data_type, StringType())
                    bronze_df = bronze_df.withColumn(column_name, col(column_name).cast(spark_data_type))
                    msg = 'Data Type Casting was Completed'
                else:
                    print(f"Column '{column_name}' does not exist in the DataFrame. Skipping...")

            column_order = [row['COLUMN_NAME'] for row in table_config_sorted.collect() if row['COLUMN_NAME'] in bronze_df.columns]
            bronze_df = bronze_df.select(*column_order)

        else:
            msg = 'Data Type Casting was Not Done'

    except TypeError as e:
        is_okay = 0
        error_message = str(e)
        msg = 'Data Type Casting Failed due to Type Error'
    except ValueError as e:
        is_okay = 0
        error_message = str(e)
        msg = 'Data Type Casting Failed due to Value Error'
    except Exception as e:
        is_okay = 0
        error_message = str(e)
        msg = 'Data Type Casting Failed due to General Error'

    return is_okay, bronze_df, msg, error_message

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 14, Finished, Available, Finished, False)

In [13]:
################################################### USER DEFINED FUNCTION: 3.ADDING WAREHOUSE AUDIT COLUMNS ####################################################
def adding_ETLWarehousing_Columns(bronze_df:DataFrame):
        is_okay = 1
        error_message = None
        msg = 'Adding ETL Warehousing Audit columns '
        current_utc_time = datetime.now(pytz.UTC)
        try:
            user_name = mssparkutils.env.getUserName()
        except Exception as e:
            user_name = "FabricDataEngineer"
        try:
            bronze_df = bronze_df.withColumn(
                                        "HashKey",
                                        md5(concat_ws("", *[col(c).cast("string") for c in bronze_df.columns]))
                                    )
            bronze_df = bronze_df.withColumn("ETLLoadedDate", lit(current_utc_time)) \
                                .withColumn("ETLLoadedBy", lit(user_name)) \
                                .withColumn("ETLModifiedDate", lit(current_utc_time)) \
                                .withColumn("ETLModifiedBy", lit(user_name)) 
            return is_okay, bronze_df, msg, error_message 
        except Exception as e:
            is_okay = 0
            error_message = str(e)
            msg= "Failed During adding warehouse Audit Columns"
            return is_okay, bronze_df, msg, error_message  

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 15, Finished, Available, Finished, False)

In [14]:
################################################### USER DEFINED FUNCTION: 4.SILVER DATA LOAD ####################################################
def data_load(bronze_df, silver_table_name, silver_schema_name):
    is_okay = 1
    error_message = ''
    msg = ''

    generic_path = silver_base_path
    relative_path = f"{silver_schema_name}/{silver_table_name}"
    absolute_path = f"{generic_path}/{relative_path}"
    
    try:
        bronze_df=bronze_df.distinct()
        bronze_df.write.mode('overwrite') \
            .option("overwriteSchema", "true") \
            .save(absolute_path)
        
        msg = "Data is loaded successfully in Silver Layer"
        return is_okay, error_message, msg
    
    except Exception as e:
        error_message = str(e)
        msg = 'Error while loading the data'
        is_okay = 0
        return is_okay, error_message, msg

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 16, Finished, Available, Finished, False)

In [15]:
################################################### USER DEFINED FUNCTION: 5.LOADING BRONZE DETAIL LOG  ####################################################
def update_table_level_log(log_dict,  EndTime=None, DurationInSec=None,
                            BronzeDataRead=None, DataTypeCasting=None, 
                            BronzeCount=None, SilverCount=None,
                            SilverDataLoad=None,
                            ErrorMessage=None, Status=None):

    if EndTime is not None:
        log_dict["EndTime"] = EndTime
    if DurationInSec is not None:
        log_dict["DurationInSec"] = DurationInSec
    if BronzeDataRead is not None:
        log_dict["BronzeDataRead"] = BronzeDataRead
    if DataTypeCasting is not None:
        log_dict["DataTypeCasting"] = DataTypeCasting
    if BronzeCount is not None:
        log_dict["BronzeCount"] = BronzeCount
    if SilverCount is not None:
        log_dict["SilverCount"] = SilverCount
    if SilverDataLoad is not None:
        log_dict["SilverDataLoad"] = SilverDataLoad
    if ErrorMessage is not None:
        log_dict["ErrorMessage"] = ErrorMessage
    if Status is not None:
        log_dict["Status"] = Status
    return log_dict

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 17, Finished, Available, Finished, False)

In [16]:
############################################  MAIN BLOCK ##################################################
def process_table(row, InformationSchema_df,ETLBatchId):
    ################################### DECLARING THE REQUIRED VARIABLES #####################################
    start_utc_time = datetime.now(timezone.utc)
    try:
        user_name = mssparkutils.env.getUserName()
    except  Exception as e:
        user_name = "FabricDataEngineer"
    failure_message='Not executed due to prior Failure'
    skip_message ="No new records were found; skip further processing"
    msg = None 
    bronze_count, silver_count, source_delete_count, updated_row_count, inserted_row_count = None, None, None, None, None
    #################################### ACCESSING THE METADATA OF THE TABLE #################################
    TableId = row['Id']
    SourceTableName = row['SourceTableName']
    SourceSchemaName = row['SourceSchemaName']
    BronzeSchemaName = row['BronzeSchemaName']
    BronzeTableName = row['BronzeTableName']
    SilverSchemaName = row["SilverSchemaName"]
    SilverTableName = row['SilverTableName']
    readtype = row['LoadType']
    ####################################### DICTIONARY FOR TABLE LEVEL LOG DETAILS ####################################
    TableLevelLog = {
                    "LogHeaderId": ETLBatchId,
                    "TableId":TableId,
                    "TableName": row['SourceTableName'],
                    "SchemaName": row['SourceSchemaName'],
                    "StartTime": start_utc_time,
                    "EndTime": '', 
                    "DurationInSec": '',
                    "BronzeDataRead": '',
                    "DataTypeCasting": '',
                    "BronzeCount":bronze_count,
                    "SilverCount":silver_count,
                    "SilverDataLoad": '',
                    "ErrorMessage": '',
                    "Status": '',
                    "ETLLoadedBy": user_name 

    }
    print(f"Started Bronze To Silver Process for SchemaName: {BronzeSchemaName} - TableName: {SourceTableName}")
    ####################################### 1. BRONZE DATA READ ##############################################
    is_okay, bronze_df, msg, error_message = Read_bronze_data( BronzeSchemaName,BronzeTableName,readtype)
    if bronze_df is None or is_okay ==0:
        TableLevelLog = update_table_level_log(
                            TableLevelLog,
                            EndTime=datetime.now(timezone.utc),
                            DurationInSec=(datetime.now(timezone.utc) - start_utc_time).total_seconds(),
                            BronzeDataRead=msg,
                            DataTypeCasting=failure_message,
                            SilverDataLoad=failure_message,
                            ErrorMessage=error_message,
                            Status="Failure"
                        )
        print(f"Error Occured while extracting the data from the bronze layer for SchemaName: {BronzeSchemaName} - TableName: {SourceTableName}")
        return TableLevelLog
    bronze_count = bronze_df.count()
    if bronze_count == 0:
        TableLevelLog = update_table_level_log(
                            TableLevelLog,
                            EndTime=datetime.now(timezone.utc),
                            DurationInSec=(datetime.now(timezone.utc) - start_utc_time).total_seconds(),
                            BronzeDataRead=msg,
                            DataTypeCasting=skip_message,
                            BronzeCount=0,
                            SilverCount = 0,
                            SilverDataLoad=skip_message,
                            ErrorMessage=None,
                            Status="Success"
                        )
        print(f"No New Record was extracting the data from the bronze layer for SchemaName: {BronzeSchemaName} - TableName: {SourceTableName}")
        return TableLevelLog

    else:
        TableLevelLog = update_table_level_log(TableLevelLog,BronzeDataRead=msg,BronzeCount= bronze_count)
        
    ####################################### 2. DATA TYPE CASTING ##############################################
    data_typecasting_df = InformationSchema_df.filter(((col("TABLE_NAME")==SourceTableName) & (col("OWNER") == SourceSchemaName ))) 
    is_okay, bronze_df, msg, error_message = data_type_casting(SourceTableName, bronze_df, data_typecasting_df)
    if(is_okay == 0):
        TableLevelLog = update_table_level_log(
                        TableLevelLog,
                        EndTime=datetime.now(timezone.utc),
                        DurationInSec=(datetime.now(timezone.utc) - start_utc_time).total_seconds(),
                        DataTypeCasting = msg,
                        SilverDataLoad=failure_message,
                        ErrorMessage=error_message,
                        Status="Failure"
                    )
        print(f"Error Occured during data type casting for SchemaName: {BronzeSchemaName} - TableName: {SourceTableName}")
        return TableLevelLog
    else:
        TableLevelLog = update_table_level_log(TableLevelLog,DataTypeCasting=msg)   
       
    ####################################### 5. ETL WAREHOUSE AUDIT COLUMNS ##############################################

    is_okay, bronze_df, msg, error_message = adding_ETLWarehousing_Columns(bronze_df)
    if(is_okay == 0):
        TableLevelLog = update_table_level_log(
                            TableLevelLog,
                            EndTime=datetime.now(timezone.utc),
                            DurationInSec=(datetime.now(timezone.utc) - start_utc_time).total_seconds(),
                            ErrorMessage=error_message,
                            Status="Failure"
            )
        print(f"The Adding Warehouse Audit Columns was failed for SchemaName : {BronzeSchemaName} - TableName: {SourceTableName}")
         
        return TableLevelLog 
    

    ####################################### 6. SILVER LAYER DATA LOAD ##############################################

    is_okay, error_message, msg = data_load(bronze_df, silver_table_name = SilverTableName, silver_schema_name= SilverSchemaName)
    if(is_okay == 0):
        TableLevelLog = update_table_level_log(
                            TableLevelLog,
                            EndTime=datetime.now(timezone.utc),
                            DurationInSec=(datetime.now(timezone.utc) - start_utc_time).total_seconds(),
                            SilverDataLoad=msg,
                            ErrorMessage=error_message,
                            Status="Failure"
            )
        print(f"Data Load was Failed for SchemaName : {BronzeSchemaName} - TableName: {SourceTableName}")
        return TableLevelLog
        
    else: 
        TableLevelLog = update_table_level_log(
                            TableLevelLog,
                            EndTime=datetime.now(timezone.utc),
                            DurationInSec=(datetime.now(timezone.utc) - start_utc_time).total_seconds(),
                            SilverDataLoad=msg,
                            SilverCount = bronze_count,
                            ErrorMessage=None,
                            Status="Success"
            )
        print(f"Data Load was Completed for SchemaName : {BronzeSchemaName} - TableName: {SourceTableName}")


    ####################################### THE END ##############################################
    return TableLevelLog


StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 18, Finished, Available, Finished, False)

In [20]:
################################################### EXECUTION BLOCK ####################################################
log_list = []
max_executors = 20
def process_row(row):
    return process_table(row, InformationSchema_df, ETLBatchId)
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_executors) as executor:
        futures = {executor.submit(process_row, row): row for row in ConfigETL_df.toLocalIterator()}
        for future in concurrent.futures.as_completed(futures):
            log_entry = future.result()
            log_list.append(log_entry)
    if not log_list:
        print("No logs generated. Check the processing function.")
        log_df = pd.DataFrame()
    else:
        log_df = pd.DataFrame(log_list)
        log_df = log_df.astype(str)

except Exception as e:
    print(f"An error occurred: {e}")
    log_df = pd.DataFrame(log_list)
    log_df = log_df.astype(str)



StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 22, Finished, Available, Finished, False)

Started Bronze To Silver Process for SchemaName: HOSPITAL - TableName: ALL_DATATYPES_TEST
File
abfss://OracleToFabricRTI@onelake.dfs.fabric.microsoft.com/LH_BRONZE.Lakehouse/Files/HOSPITAL/ALL_DATATYPES_TEST/*.parquet
Started Bronze To Silver Process for SchemaName: HOSPITAL - TableName: ALL_DATATYPES_TEST
File
abfss://OracleToFabricRTI@onelake.dfs.fabric.microsoft.com/LH_BRONZE.Lakehouse/Files/HOSPITAL/ALL_DATATYPES_TEST/*.parquet
Data REad
Data REad
Data read completed for ALL_DATATYPES_TEST
Data read completed for ALL_DATATYPES_TEST
Column 'ID' does not exist in the DataFrame. Skipping...
Column 'CHAR_COL' does not exist in the DataFrame. Skipping...
Column 'VARCHAR_COL' does not exist in the DataFrame. Skipping...
Column 'NCHAR_COL' does not exist in the DataFrame. Skipping...
Column 'NVARCHAR_COL' does not exist in the DataFrame. Skipping...
Column 'NUMBER_COL' does not exist in the DataFrame. Skipping...
Column 'NUMBER_PRECISION_COL' does not exist in the DataFrame. Skipping...
C

In [21]:
################################################### SAVING LOG DETAILS ####################################################
schema = StructType([
    StructField("LogHeaderId", StringType(), True),
    StructField("TableId", StringType(), True),
    StructField("SchemaName", StringType(), True),
    StructField("TableName", StringType(), True),
    StructField("StartTime", StringType(), True),
    StructField("EndTime", StringType(), True),
    StructField("DurationInSec", StringType(), True),
    StructField("BronzeDataRead", StringType(), True),
    StructField("DataTypeCasting", StringType(), True),
    StructField("BronzeCount", StringType(), True),
    StructField("SilverCount", StringType(), True),
    StructField("SilverDataLoad", StringType(), True),
    StructField("ErrorMessage", StringType(), True),
    StructField("Status", StringType(), True),
    StructField("EtlLoadedBy", StringType(), True)
])
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
################################################### SAVING LOG DETAILS ####################################################
spark_df = spark.createDataFrame(log_df, schema=schema)
spark_df = spark_df.withColumn("BronzeCount", col("BronzeCount").cast(LongType()).cast(StringType()))
spark_df = spark_df.withColumn("SilverCount", col("SilverCount").cast(LongType()).cast(StringType()))
spark_df = spark_df.withColumnRenamed("LogHeaderId","BatchId")
# spark_df.write.option("overwriteSchema","true").mode("append").synapsesql("WH_MetaData.LOG.ETLBatchSilverLogDetails")

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 23, Finished, Available, Finished, False)

In [22]:
display(spark_df)

StatementMeta(, 1501fbe6-195b-4dd8-95c3-a44efb36a35c, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3685f088-560f-4034-90c6-043393c410b5)